# ML and DL

## Dependencies

In [ ]:
import os
import random
import sys
import warnings

import dill
import numpy as np
import pandas as pd
import torch
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [ ]:
warnings.filterwarnings("ignore")  # 忽略警告
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei"]  # 用来正常显示中文标签
plt.rcParams["axes.unicode_minus"] = False  # 用来正常显示负号

In [ ]:
root_dir = "C:\\ML4GM"
outputs_dir = os.path.join(root_dir, "models")  # 输出路径
os.makedirs(outputs_dir, exist_ok=True)  # 创建数据目录

In [ ]:
def save_pkl(filepath, data):
    # 保存模型
    with open(filepath, "wb") as fw:
        dill.dump(data, fw)
    print(f"[{filepath}] data saving...")

## Load and proc

In [ ]:
data = pd.read_csv("C:\\ML4GM\\proc_data\\02_merge\\merged_data_cleaned.csv")  # 加载数据

data

In [ ]:
# 保留 data 含 rgiid/year 供后续按组划分；EDA 使用不含 ID 的 data_eda
data_eda = data.drop(columns=["rgiid", "year", "GLIMSId"], axis=1)
data_eda

### 缺失值统计与处理

In [ ]:
# 缺失值统计
missing_count = data.isnull().sum()
missing_pct = (data.isnull().sum() / len(data) * 100).round(2)
missing_df = pd.DataFrame({"缺失数": missing_count, "缺失占比%": missing_pct})
print("各列缺失值统计：")
print(missing_df[missing_df["缺失数"] > 0] if missing_count.sum() > 0 else "无缺失值，无需处理。")

if missing_count.sum() > 0:
    # 目标列 dhdt：删除目标为缺失的行
    data = data.dropna(subset=["dhdt"])
    # 特征列：按列用中位数填充
    data = data.fillna(data.median(numeric_only=True))
    print("已处理：目标缺失行已删除，特征缺失已用中位数填充。")

### Correlation

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(14, 12), dpi=100)  # 定义画布
sns.heatmap(
    data_eda.corr(),
    annot=True,
    fmt=".2g",
    cmap="RdBu",
    annot_kws={"size": 8, "weight": "bold"},
    ax=ax,
)  # 绘制相关性图
ax.set_title("Correlation coefficient heatmap", fontsize=20)  # 标题
ax.set_xlabel("Features", fontsize=12)  # x轴标签
ax.set_ylabel("Features", fontsize=12)  # y轴标签
ax.tick_params(labelsize=10)  # 设置坐标轴轴刻度大小
plt.xticks(rotation=45, ha="right")
plt.tight_layout()  # 防重叠
plt.show()  # 显示图像
plt.close()  # 关闭图像

### 描述性统计

In [ ]:
data_eda.describe()  # 描述性统计

### 目标与主要特征分布

In [ ]:
# 扩展 preprocessed_data.pkl：增加全量数据与分组信息（用于 nested GroupKFold）
from pathlib import Path

pkl_path = os.path.join(outputs_dir, "preprocessed_data.pkl")
if Path(pkl_path).exists():
    _data_loaded = load_pkl(pkl_path) if "load_pkl" in globals() else dill.load(open(pkl_path, "rb"))

    # 依赖于前文已定义的 data/X/y 与 train/val/test 掩码
    _data_loaded["X_all"] = X
    _data_loaded["y_all"] = y

    # rgiid 分组（空间泛化）
    _data_loaded["rgiid_all"] = data["rgiid"].values
    _data_loaded["rgiid_train"] = data.loc[train_mask, "rgiid"].values
    _data_loaded["rgiid_val"] = data.loc[val_mask, "rgiid"].values
    _data_loaded["rgiid_test"] = data.loc[test_mask, "rgiid"].values

    # year 分组（时间外推/泛化）
    _data_loaded["year_all"] = data["year"].values
    _data_loaded["year_train"] = data.loc[train_mask, "year"].values
    _data_loaded["year_val"] = data.loc[val_mask, "year"].values
    _data_loaded["year_test"] = data.loc[test_mask, "year"].values

    save_pkl(pkl_path, _data_loaded)
    print("Updated preprocessed_data.pkl with X_all/y_all and rgiid/year groups for nested CV.")
else:
    print("preprocessed_data.pkl not found; run the previous cell to generate it first.")

In [ ]:
# 目标变量 dhdt 分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data_eda["dhdt"], kde=True, ax=axes[0], bins=50)
axes[0].set_title("目标变量 dhdt 分布")
axes[0].set_xlabel("dhdt")

# 主要特征分布（选取代表性列）
main_cols = ["dhdt", "Area", "Zmed", "Slope", "Lmax"]
data_eda[main_cols].hist(bins=40, figsize=(12, 8), layout=(2, 3), edgecolor="black", alpha=0.7)
plt.suptitle("目标与主要特征分布", fontsize=14)
plt.tight_layout()
plt.show()
plt.close()

### 异常值检查（可选）

In [ ]:
# IQR 异常值统计（仅检查，不删除）
check_cols = ["dhdt", "Area", "Zmed", "Lmax"]
for col in check_cols:
    Q1 = data_eda[col].quantile(0.25)
    Q3 = data_eda[col].quantile(0.75)
    IQR = Q3 - Q1
    low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((data_eda[col] < low) | (data_eda[col] > high)).sum()
    pct = (n_out / len(data_eda) * 100).round(2)
    print(f"{col}: 异常值数量={n_out}, 占比={pct}%")

# 关键变量箱线图
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
data_eda[["dhdt", "Area", "Zmed"]].boxplot(ax=ax)
ax.set_title("关键变量箱线图（异常值可视化）")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
plt.close()

### set x&y

In [ ]:
feature_columns = data_eda.columns.tolist()[1:]  # 特征列
target_columns = data_eda.columns.tolist()[0]  # 目标列

X = data_eda[feature_columns].values  # 特征
y = data_eda[target_columns].values  # 目标

print(f"X shape: {X.shape}, y shape: {y.shape}")  # 打印维度

### 划分策略（按组/时间，避免泄漏）

泛化对象：**新冰川**（按 `rgiid` 划分）或 **新年份**（按 `year` 划分），保证同一冰川或同一年不同时出现在训练集与测试集中。

In [ ]:
# 按组划分：train/val/test = 80/10/10，避免同一组出现在多份数据中
split_by = "rgiid"  # 可选 "year"
train_ratio, val_ratio, test_ratio = 0.8, 0.1, 0.1
rs = np.random.RandomState(42)

unique_groups = data[split_by].unique()
rs.shuffle(unique_groups)
n = len(unique_groups)
n_train = max(1, int(n * train_ratio))
n_val = max(0, int(n * val_ratio))
n_test = max(1, n - n_train - n_val)
train_groups = set(unique_groups[:n_train])
val_groups = set(unique_groups[n_train : n_train + n_val])
test_groups = set(unique_groups[n_train + n_val :])

train_mask = data[split_by].isin(train_groups).values
val_mask = data[split_by].isin(val_groups).values
test_mask = data[split_by].isin(test_groups).values

X_train = X[train_mask]
X_val = X[val_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_val = y[val_mask]
y_test = y[test_mask]

print(f"划分依据: {split_by}, 训练组数: {len(train_groups)}, 验证组数: {len(val_groups)}, 测试组数: {len(test_groups)}")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

### 数据标准化

In [ ]:
X_scaler = StandardScaler()  # 定义标准化模型
X_train = X_scaler.fit_transform(X_train)  # 训练集标准化
X_val = X_scaler.transform(X_val)  # 验证集标准化
X_test = X_scaler.transform(X_test)  # 测试集标准化

save_pkl(os.path.join(outputs_dir, "X_scaler.pkl"), X_scaler)  # 保存模型

In [ ]:
y_scaler = StandardScaler()  # 定义标准化模型
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1)).reshape(-1)  # 训练集标准化
y_val = y_scaler.transform(y_val.reshape(-1, 1)).reshape(-1)  # 验证集标准化（与训练/测试保持一致）
y_test = y_scaler.transform(y_test.reshape(-1, 1)).reshape(-1)  # 测试集标准化

save_pkl(os.path.join(outputs_dir, "y_scaler.pkl"), y_scaler)  # 保存模型

In [ ]:
# --- Save preprocessed data for 02-07 ---
_data = {
    "X_train": X_train, "X_val": X_val, "X_test": X_test,
    "y_train": y_train, "y_val": y_val, "y_test": y_test,
    "X_scaler": X_scaler, "y_scaler": y_scaler,
    "feature_columns": feature_columns,
    # Minimal metadata for reproducibility / consistent semantics
    "split_by": split_by,
    "train_ratio": train_ratio,
    "val_ratio": val_ratio,
    "test_ratio": test_ratio,
    "random_state": 42,
    "y_standardized": True,
}
save_pkl(os.path.join(outputs_dir, "preprocessed_data.pkl"), _data)
print("Preprocessed data saved to", os.path.join(outputs_dir, "preprocessed_data.pkl"))

In [ ]:
print("pkl keys:", list(_data.keys()))
print("X_val/y_val:", "X_val" in _data, "y_val" in _data)